In [5]:
from google.colab import files
uploaded = files.upload()


Saving train.csv to train.csv


In [6]:
import os
print(os.listdir())

['.config', 'archive.zip', 'train.csv', 'sample_data']


In [7]:
import pandas as pd

df = pd.read_csv("train.csv")

print(df.shape)
print(df.head())
print(df.info())
print(df.isna().sum())



(45593, 20)
        ID Delivery_person_ID Delivery_person_Age Delivery_person_Ratings  \
0  0x4607     INDORES13DEL02                   37                     4.9   
1  0xb379     BANGRES18DEL02                   34                     4.5   
2  0x5d6d     BANGRES19DEL01                   23                     4.4   
3  0x7a6a    COIMBRES13DEL02                   38                     4.7   
4  0x70a2     CHENRES12DEL01                   32                     4.6   

   Restaurant_latitude  Restaurant_longitude  Delivery_location_latitude  \
0            22.745049             75.892471                   22.765049   
1            12.913041             77.683237                   13.043041   
2            12.914264             77.678400                   12.924264   
3            11.003669             76.976494                   11.053669   
4            12.972793             80.249982                   13.012793   

   Delivery_location_longitude  Order_Date Time_Orderd Time_Order_pi

In [8]:

import numpy as np

# Remove extra spaces from column names and text values
df.columns = df.columns.str.strip()
for c in df.select_dtypes(include="object").columns:
    df[c] = df[c].str.strip().replace("NaN", np.nan)

# Remove "conditions " prefix from weather
df["Weatherconditions"] = (df["Weatherconditions"]
                           .str.replace("conditions ", "", regex=False)
                           .replace("NaN", np.nan))

# Time_taken looks like "(min) 24", so keep only the number
df["Time_taken(min)"] = df["Time_taken(min)"].str.extract(r"(\d+)").astype(float)

# Convert numeric columns that were stored as text
for c in ["Delivery_person_Age", "Delivery_person_Ratings", "multiple_deliveries"]:
    df[c] = pd.to_numeric(df[c], errors="coerce")

print(df.isna().sum())


ID                                0
Delivery_person_ID                0
Delivery_person_Age            1854
Delivery_person_Ratings        1908
Restaurant_latitude               0
Restaurant_longitude              0
Delivery_location_latitude        0
Delivery_location_longitude       0
Order_Date                        0
Time_Orderd                    1731
Time_Order_picked                 0
Weatherconditions               616
Road_traffic_density            601
Vehicle_condition                 0
Type_of_order                     0
Type_of_vehicle                   0
multiple_deliveries             993
Festival                        228
City                           1200
Time_taken(min)                   0
dtype: int64


In [9]:
log = []

# Ratings above 5 are impossible
n = (df["Delivery_person_Ratings"] > 5).sum()
df.loc[df["Delivery_person_Ratings"] > 5, "Delivery_person_Ratings"] = np.nan
log.append(["Ratings > 5", n, "Set to missing", "Rating scale is 1-5"])

# Ages under 18 are not valid delivery partners
n = (df["Delivery_person_Age"] < 18).sum()
df.loc[df["Delivery_person_Age"] < 18, "Delivery_person_Age"] = np.nan
log.append(["Age < 18", n, "Set to missing", "Below legal working age"])

# Negative coordinates: flip sign, since India has positive lat/long
for c in ["Restaurant_latitude", "Restaurant_longitude",
          "Delivery_location_latitude", "Delivery_location_longitude"]:
    df[c] = df[c].abs()

# Zero coordinates are placeholders, so drop those rows
bad = (df["Restaurant_latitude"] < 1) | (df["Restaurant_longitude"] < 1)
log.append(["Restaurant coords ~0", bad.sum(), "Dropped rows", "Not real locations"])
df = df[~bad]

# Rows with no delivery time cannot be analyzed
n = df["Time_taken(min)"].isna().sum()
log.append(["Missing Time_taken", n, "Dropped rows", "Target column"])
df = df.dropna(subset=["Time_taken(min)"])

# Convert date and remove duplicate orders
df["Order_Date"] = pd.to_datetime(df["Order_Date"], format="%d-%m-%Y")
n = df.duplicated(subset="ID").sum()
df = df.drop_duplicates(subset="ID")
log.append(["Duplicate IDs", n, "Dropped", "One row per order"])

cleaning_log = pd.DataFrame(log, columns=["Issue", "Rows affected", "Action", "Why"])
print(cleaning_log)
print(df.shape)

                  Issue  Rows affected          Action  \
0           Ratings > 5             53  Set to missing   
1              Age < 18             38  Set to missing   
2  Restaurant coords ~0           3640    Dropped rows   
3    Missing Time_taken              0    Dropped rows   
4         Duplicate IDs              0         Dropped   

                       Why  
0      Rating scale is 1-5  
1  Below legal working age  
2       Not real locations  
3            Target column  
4        One row per order  
(41953, 20)


In [11]:
df.to_csv("clean_deliveries.csv", index=False)
cleaning_log.to_csv("cleaning_log.csv", index=False)

from google.colab import files
files.download("clean_deliveries.csv")
files.download("cleaning_log.csv")

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

In [12]:
df = df.rename(columns={"City": "area_type"})

def haversine(lat1, lon1, lat2, lon2):
    R = 6371
    lat1, lon1, lat2, lon2 = map(np.radians, [lat1, lon1, lat2, lon2])
    a = np.sin((lat2 - lat1) / 2) ** 2 + np.cos(lat1) * np.cos(lat2) * np.sin((lon2 - lon1) / 2) ** 2
    return 2 * R * np.arcsin(np.sqrt(a))

df["distance_km"] = haversine(df["Restaurant_latitude"], df["Restaurant_longitude"],
                              df["Delivery_location_latitude"], df["Delivery_location_longitude"])
print(df["distance_km"].describe())

count    41953.000000
mean         9.720284
std          5.603083
min          1.465067
25%          4.657655
50%          9.193021
75%         13.680920
max         20.969489
Name: distance_km, dtype: float64


In [13]:
t_order = pd.to_datetime(df["Time_Orderd"], format="%H:%M:%S", errors="coerce")
t_pick = pd.to_datetime(df["Time_Order_picked"], format="%H:%M:%S", errors="coerce")

prep = (t_pick - t_order).dt.total_seconds() / 60
prep = prep.where(prep >= 0, prep + 1440)   # fixes orders that cross midnight
df["prep_time_min"] = prep

df["order_hour"] = t_order.dt.hour
df["day_of_week"] = df["Order_Date"].dt.day_name()

def slot(h):
    if pd.isna(h): return np.nan
    if 5 <= h <= 10: return "Morning"
    if 11 <= h <= 14: return "Lunch"
    if 15 <= h <= 17: return "Evening snack"
    if 18 <= h <= 22: return "Dinner"
    return "Late night"

df["time_slot"] = df["order_hour"].apply(slot)
print(df[["prep_time_min", "order_hour", "time_slot"]].describe(include="all"))

        prep_time_min    order_hour time_slot
count    40353.000000  40353.000000     40353
unique            NaN           NaN         5
top               NaN           NaN    Dinner
freq              NaN           NaN     21071
mean         9.978688     17.421406       NaN
std          4.087331      4.815816       NaN
min          5.000000      0.000000       NaN
25%          5.000000     15.000000       NaN
50%         10.000000     19.000000       NaN
75%         15.000000     21.000000       NaN
max         15.000000     23.000000       NaN


In [14]:
threshold = df["Time_taken(min)"].quantile(0.75)
print("Late threshold (minutes):", threshold)

df["is_late"] = (df["Time_taken(min)"] > threshold).astype(int)
df["distance_bucket"] = pd.cut(df["distance_km"], bins=[0, 3, 6, 10, 100],
                               labels=["0-3 km", "3-6 km", "6-10 km", "10+ km"])

Late threshold (minutes): 32.0


In [15]:
print(df.groupby("Road_traffic_density")["is_late"].mean().round(3))
print(df.groupby("multiple_deliveries")["Time_taken(min)"].mean().round(1))
print(df.groupby("time_slot")["is_late"].mean().round(3))
print(df.groupby("distance_bucket", observed=True)["is_late"].mean().round(3))

Road_traffic_density
High      0.245
Jam       0.440
Low       0.064
Medium    0.258
Name: is_late, dtype: float64
multiple_deliveries
0.0    22.9
1.0    26.9
2.0    40.5
3.0    47.9
Name: Time_taken(min), dtype: float64
time_slot
Dinner           0.344
Evening snack    0.238
Late night       0.101
Lunch            0.234
Morning          0.012
Name: is_late, dtype: float64
distance_bucket
0-3 km     0.072
3-6 km     0.100
6-10 km    0.147
10+ km     0.401
Name: is_late, dtype: float64


In [17]:
df.to_csv("featured_deliveries.csv", index=False)
files.download("featured_deliveries.csv")

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>